# Churn prediction — Kaggle submission notebook (improved)

This notebook focuses on **Kaggle-safe generalization**:
- Drops potential leakage/ID column (`Unnamed: 0`) from features (keeps it only as row id for submission).
- Uses **StratifiedKFold** CV.
- Trains a small **ensemble** (LightGBM + XGBoost + HistGradientBoosting when available).
- Produces **both**:
  - `results_proba.csv` — probability predictions (submit if Kaggle metric is ROC-AUC / LogLoss)
  - `results.csv` — class predictions using an **optimized MCC threshold** (submit if Kaggle metric expects class labels)


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

RANDOM_STATE = 42
N_SPLITS = 5

TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"

TARGET_COL = "target_class"
ID_COL = "Unnamed: 0"  # present in provided files; used as id in the submission


In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

assert TARGET_COL in train.columns, f"Expected target column '{TARGET_COL}' not found."
assert TARGET_COL not in test.columns, "test.csv unexpectedly contains the target column."

# Basic sanity checks
print("train shape:", train.shape)
print("test  shape:", test.shape)
print("target distribution:\n", train[TARGET_COL].value_counts(normalize=True).rename("share"))

# Verify the ID column exists; if not, we will fall back to the row index
use_id = ID_COL in train.columns and ID_COL in test.columns
print("Using ID column:", ID_COL if use_id else "<row_index>")


train shape: (24000, 49)
test  shape: (6000, 48)
target distribution:
 target_class
1    0.8
0    0.2
Name: share, dtype: float64
Using ID column: Unnamed: 0


In [3]:
y = train[TARGET_COL].astype(int)

drop_cols = [TARGET_COL]
if use_id:
    drop_cols.append(ID_COL)

X = train.drop(columns=drop_cols)
X_test = test.drop(columns=[ID_COL]) if use_id else test.copy()

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

# No missing values in provided data, but keep this for robustness:
assert X.isna().sum().max() == 0 and X_test.isna().sum().max() == 0, "Unexpected missing values found."


X shape: (24000, 47)
X_test shape: (6000, 47)


## Models

We try to use **LightGBM** and **XGBoost** if installed (often best on tabular data).
If they are not available in your environment, we fall back to sklearn-only models so the notebook still runs.


In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Optional dependencies (graceful fallback)
HAS_LGBM = False
HAS_XGB  = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    print("LightGBM not available -> will skip LGBM. Error:", str(e)[:120])

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    print("XGBoost not available -> will skip XGB. Error:", str(e)[:120])

def make_models(class_weight_balanced=True):
    models = []

    if HAS_LGBM:
        models.append((
            "lgbm",
            LGBMClassifier(
                n_estimators=4000,
                learning_rate=0.02,
                num_leaves=31,
                min_child_samples=30,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.5,
                reg_lambda=0.5,
                class_weight="balanced" if class_weight_balanced else None,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        ))

    if HAS_XGB:
        # compute scale_pos_weight for imbalance: neg/pos
        pos = (y == 1).sum()
        neg = (y == 0).sum()
        spw = float(neg) / float(pos) if pos else 1.0

        models.append((
            "xgb",
            XGBClassifier(
                n_estimators=5000,
                learning_rate=0.02,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.5,
                reg_lambda=1.0,
                min_child_weight=1.0,
                gamma=0.0,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=spw,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            )
        ))

    # Strong sklearn baseline that doesn't need extra libs
    models.append((
        "hgb",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=None,
            max_iter=600,
            l2_regularization=0.5,
            random_state=RANDOM_STATE
        )
    ))

    return models

models = make_models()
print("Models:", [name for name, _ in models])


Models: ['lgbm', 'xgb', 'hgb']


## Cross-validation + out-of-fold predictions

We compute out-of-fold (OOF) probabilities for each model, then **average** them (simple ensemble).
We also search the **best threshold** for MCC on OOF predictions (if Kaggle expects class labels).


In [6]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_pred = {name: np.zeros(len(X), dtype=float) for name, _ in models}
best_iters = {name: [] for name, _ in models}

def fit_predict_fold(model_name, model, X_tr, y_tr, X_va, y_va):
    # LightGBM and XGBoost benefit from early stopping
    if model_name == "lgbm":
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            callbacks=[
                __import__("lightgbm").early_stopping(stopping_rounds=200, verbose=False)
            ]
        )
        best_iters[model_name].append(int(model.best_iteration_))
        return model.predict_proba(X_va)[:, 1]

    if model_name == "xgb":
        import xgboost as xgb

        # Some xgboost versions don't support early_stopping_rounds in .fit()
        # Use callbacks when available, otherwise fall back to plain fit.
        try:
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                verbose=False,
                callbacks=[xgb.callback.EarlyStopping(rounds=200, save_best=True)]
            )
        except TypeError:
            # fallback: no early stopping
            model.fit(X_tr, y_tr)

        # best_iteration handling differs by version
        best_it = getattr(model, "best_iteration", None)
        if best_it is None:
            # older variants: best_iteration can be missing; use n_estimators as proxy
            best_it = getattr(model, "n_estimators", None)

        if best_it is not None:
            best_iters[model_name].append(int(best_it))

        return model.predict_proba(X_va)[:, 1]

    # sklearn models
    model.fit(X_tr, y_tr)
    # HistGradientBoostingClassifier supports predict_proba
    return model.predict_proba(X_va)[:, 1]

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

    for name, base_model in models:
        # clone by re-instantiating (safe enough here)
        model = make_models()[[n for n,_ in models].index(name)][1]
        proba = fit_predict_fold(name, model, X_tr, y_tr, X_va, y_va)
        oof_pred[name][va_idx] = proba

    # quick fold summary using ensemble mean so far
    ens_va = np.mean([oof_pred[n][va_idx] for n,_ in models], axis=0)
    auc = roc_auc_score(y_va, ens_va)
    print(f"Fold {fold}/{N_SPLITS}: ensemble AUC={auc:.5f}")

# OOF ensemble
oof_ens = np.mean([oof_pred[n] for n,_ in models], axis=0)

# Metrics on OOF
oof_auc = roc_auc_score(y, oof_ens)
oof_acc = accuracy_score(y, (oof_ens >= 0.5).astype(int))
print("\nOOF ensemble metrics (threshold=0.5):")
print("  AUC :", round(oof_auc, 6))
print("  ACC :", round(oof_acc, 6))


[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000985 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5941
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fold 1/5: ensemble AUC=0.96132
[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5916
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fold 2/5: ensemble AUC=0.95871
[LightGBM] [Info] Number of positive: 15360, number of negativ

In [7]:
# Threshold optimization for MCC (useful if Kaggle expects hard class labels)
thresholds = np.linspace(0.05, 0.95, 181)
mccs = []

for t in thresholds:
    mccs.append(matthews_corrcoef(y, (oof_ens >= t).astype(int)))

best_idx = int(np.argmax(mccs))
best_t = float(thresholds[best_idx])
best_mcc = float(mccs[best_idx])

print("Best OOF MCC:", round(best_mcc, 6), "at threshold:", round(best_t, 4))


Best OOF MCC: 0.876261 at threshold: 0.52


## Train on full data + create submissions

We refit each model on **all training data** using a conservative number of estimators.
For LGBM/XGB we use the **average best_iteration** found during CV (if available).


In [8]:
final_models = {}

for name, model in make_models():
    if name in best_iters and len(best_iters[name]) > 0:
        avg_best = int(np.mean(best_iters[name]))
        if name == "lgbm":
            model.set_params(n_estimators=max(200, avg_best))
        if name == "xgb":
            model.set_params(n_estimators=max(200, avg_best))

    model.fit(X, y)
    final_models[name] = model
    print(f"{name}: fitted. n_estimators=", getattr(model, "n_estimators", None))

# Predict probabilities on test and ensemble-average
test_probas = np.mean([final_models[n].predict_proba(X_test)[:, 1] for n in final_models], axis=0)

print("test_probas stats:", pd.Series(test_probas).describe()[['min','max','mean']])


[LightGBM] [Info] Number of positive: 19200, number of negative: 4800
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.189098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6170
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
lgbm: fitted. n_estimators= 3783
xgb: fitted. n_estimators= 5000
hgb: fitted. n_estimators= None
test_probas stats: min     0.000004
max     0.999502
mean    0.792226
dtype: float64


In [9]:
# Build submission frames
if use_id:
    ids = test[ID_COL].values
else:
    ids = np.arange(len(test))

# 1) Probabilities (submit if Kaggle metric is ROC-AUC / LogLoss)
sub_proba = pd.DataFrame({ID_COL if use_id else "id": ids, TARGET_COL: test_probas})
sub_proba.to_csv("results_proba.csv", index=False)

# 2) Class labels using MCC-optimized threshold (submit if Kaggle expects labels)
sub_label = pd.DataFrame({ID_COL if use_id else "id": ids, TARGET_COL: (test_probas >= best_t).astype(int)})
sub_label.to_csv("results.csv", index=False)

print("Saved:")
print(" - results_proba.csv (probabilities)")
print(" - results.csv       (class labels, threshold =", round(best_t, 4), ")")


Saved:
 - results_proba.csv (probabilities)
 - results.csv       (class labels, threshold = 0.52 )


### Which file should you submit?

- If the competition metric is **ROC-AUC** or **LogLoss** → submit `results_proba.csv`.
- If the competition metric is **MCC / F1 / Accuracy** and expects hard labels → submit `results.csv`.
